# Taylor/MSKCC: reproducible expression and clinical data

This report reads the frozen **GSE21034 / GPL10264 / prad_mskcc** outputs and raw-source manifest. It makes no network requests. Counts are calculated from these files; historical counts are comparisons, never selection targets.

Run `uv run --locked taylor-gse21034` from the repository root before executing this notebook.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
SNAPSHOT = 'taylor-v2'
RAW = ROOT / 'data/raw' / SNAPSHOT
OUT = ROOT / 'data/processed' / SNAPSHOT
INTERIM = ROOT / 'data/interim' / SNAPSHOT
manifest = json.loads((RAW / 'manifest.json').read_text())
metadata = json.loads((OUT / 'metadata.json').read_text())
validation = json.loads((OUT / 'validation.json').read_text())
expression = pd.read_csv(OUT / 'expression.csv.gz', index_col=0, dtype={'feature_id': 'string', 'gene_id': 'string'}, float_precision='round_trip')
clinical = pd.read_csv(OUT / 'clinical.csv')
samples = pd.read_csv(OUT / 'samples.csv')
genes = pd.read_csv(OUT / 'genes.csv', dtype={'feature_id': 'string', 'gene_id': 'string'})
mapping = pd.read_csv(OUT / 'feature_mapping.csv.gz', dtype={'feature_id': 'string', 'candidate_gene_id': 'string'})
audit = pd.read_csv(OUT / 'sample_audit.csv')
pd.set_option('display.max_colwidth', 100)

## 1. Sources and integrity

Original GEO, GPL, clinical pages, clinical definitions and MyGene responses are cached. Every request has a URL, method/body, retrieval timestamp and SHA-256 in the manifest. The current snapshot downloaded every input automatically, including the GEO matrix. All 58 files have retrieval timestamps and checksums. The table below groups cached files only for readability; the full manifest retains individual requests.

In [ ]:
sources = pd.DataFrame(manifest['sources'].values())
display(sources[['path', 'method', 'retrieved_at_utc', 'bytes', 'sha256']])
def digest(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()
assert all(digest(RAW / entry['path']) == entry['sha256'] for entry in manifest['sources'].values())
assert digest(RAW / 'manifest.json') == metadata['source_manifest_sha256']
for line in (OUT / 'checksums.sha256').read_text().splitlines():
    expected, filename = line.split('  ', 1)
    assert digest(OUT / filename) == expected, filename
print('All raw and final file checksums verified.')

## 2. Population and patient matching

Select the GEO `Primary tumor` label and confirm available cBio primary labels. Exact sample IDs link GEO to cBio; the actual cBio patient association is checked against the downloaded sample inventory. Sample and patient identifiers remain separate even when their text happens to match. Missing tissue labels remain missing. Conflicting joins or multiple primary profiles for one patient require review.

In [ ]:
display(audit.groupby(['selected', 'selection_reason'], dropna=False).size().rename('profiles').to_frame())
display(samples.head(8))
print('Distinct selected profiles:', samples.sample_id.nunique())
print('Distinct selected patients:', samples.patient_id.nunique())
print('Identical clinical records collapsed:', {k:v for k,v in validation.items() if 'collapsed' in k})

## 3. Expression processing and scale

The source describes RMA background adjustment and quantile normalization. The exact deposited log base remains unverified. The pipeline preserves deposited values, adds no normalization or log transformation, and rejects missing/non-finite values. Mapped features remain separate with their original deposited values; these outputs do not establish suitability for a particular downstream scoring method.

In [ ]:
display(pd.Series(metadata['expression'], name='policy').to_frame())
features = pd.read_csv(INTERIM / 'feature_expression.csv.gz', index_col=0, dtype={'feature_id': 'string'}, float_precision='round_trip')
display(pd.DataFrame({'deposited_features': np.quantile(features.values, [0,.25,.5,.75,1]),
                      'retained_features': np.quantile(expression.values, [0,.25,.5,.75,1])},
                     index=['minimum','25%','median','75%','maximum']))
print('Feature matrix:', features.shape, '; retained feature matrix:', expression.shape)

## 4. Feature → RefSeq → gene decisions

GPL10264 supplies the original feature-to-RefSeq dictionary. MyGene candidates are cached in full. A selected candidate must be human, agree on the exact version-stripped RefSeq RNA, and have one consistent Entrez/symbol identity. Ambiguous and unmapped features stay in the audit. Gene symbols and Entrez IDs are annotations and may repeat; original feature IDs are the expression row keys. The example verifies separate features sharing a gene against the source matrix.

In [ ]:
display(mapping.drop_duplicates('feature_id').mapping_status.value_counts().rename('features').to_frame())
display(mapping.head(8))
example = genes.loc[genes.gene_id.duplicated(False)].iloc[0]
contributors = genes.loc[genes.gene_id.eq(example.gene_id), 'feature_id'].tolist()
np.testing.assert_array_equal(features.loc[contributors].to_numpy(), expression.loc[contributors].to_numpy())
display(genes.loc[genes.gene_id.eq(example.gene_id)])
display(expression.loc[contributors].iloc[:, :5])


## 5. DFS and clinical covariates

Original patient/sample fields are preserved with `patient__` and `sample__` prefixes. DFS remains DFS. The downloaded definitions specify months since initial treatment; a surgery-specific origin and exact recurrence/death rules remain unverified. A source-coded disease-free observation is censored at its recorded endpoint time, not a guarantee of lifelong absence of recurrence. Missing values are never converted to non-events.

In [ ]:
display(pd.DataFrame(metadata['dfs']['definitions']).T[['displayName', 'description', 'datatype']])
display(clinical[['sample_id','patient_id','patient__DFS_STATUS','patient__DFS_MONTHS','dfs_event','dfs_months','dfs_eligible']].head(8))
display(clinical.groupby('dfs_status', dropna=False).agg(profiles=('sample_id','size'),
    eligible=('dfs_eligible','sum'), median_observed_months=('dfs_months','median')))
print('Median observed time mixes event and censoring times; it is not a survival median.')
display(clinical.filter(regex='^(patient__|sample__)').isna().sum().rename('missing_values').to_frame())

## 6. Alignment, validation and baseline comparison

The expression columns, samples and clinical rows must agree in exact order. Gene annotations must agree with expression rows. This report verifies those invariants independently of the saved validation report.

In [ ]:
checks = {
    'sample_order': expression.columns.tolist() == samples.sample_id.tolist() == clinical.sample_id.tolist(),
    'feature_order': expression.index.astype(str).tolist() == genes.feature_id.tolist(),
    'unique_sample_ids': samples.sample_id.is_unique,
    'unique_patient_ids': samples.patient_id.is_unique,
    'unique_feature_ids': expression.index.is_unique,
    'finite_expression': bool(np.isfinite(expression.values).all()),
    'every_source_feature_audited': set(features.index.astype(str)) == set(mapping.feature_id),
    'selected_samples_audited': set(audit.loc[audit.selected, 'sample_id']) == set(samples.sample_id),
    'eligible_outcomes_complete': bool(clinical.loc[clinical.dfs_eligible, ['dfs_event','dfs_months']].notna().all().all()),
    'eligible_times_positive': bool(clinical.loc[clinical.dfs_eligible, 'dfs_months'].gt(0).all()),
}
assert all(checks.values()), checks
display(pd.Series(checks, name='passed').to_frame())
display(pd.DataFrame({'current': validation['counts'], 'legacy': validation['legacy_baseline'],
                      'difference': validation['baseline_differences']}))

In [ ]:
import anndata as ad
adata = ad.read_h5ad(ROOT / 'data/anndata' / SNAPSHOT / 'expression.h5ad')
assert adata.obs_names.tolist() == expression.columns.tolist()
assert adata.var_names.tolist() == expression.index.astype(str).tolist()
np.testing.assert_array_equal(adata.X, expression.to_numpy().T)
display(adata)
display(adata.var.head())


### Comparison with earlier gene averages

The previous build collapsed features sharing an Entrez ID. The current build retains each mapped feature, so feature rows can exceed unique genes. In particular, features `10782` and `18382` both map to `84301 / DDI2` and remain separate. Historical gene counts are compared with unique Entrez IDs, not feature counts.


## Remaining interpretation limits

Structural checks pass independently of whether the assay is ready for a specific analysis. The exact deposited log base and precise DFS recurrence criteria remain unresolved. No raw CEL quality control, additional normalization, or downstream modeling is performed here. The manifest and mapping/sample audits preserve the evidence needed to review those decisions.

In [ ]:
display(pd.Series(metadata['limitations'], name='documented_limitations').to_frame())
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in sorted(OUT.iterdir())]))